# Trees

Notes and runnable examples on trees — especially the **binary search tree (BST)** — and the Python-specific reality: there is **no built-in tree**, a naive BST quietly degrades to O(n), and for most real work you reach for `bisect` on a sorted list (or a `dict`/`set` when you don't need order).

**Contents**

1. **What it is** — terminology, the BST invariant
2. **A binary search tree from scratch**
3. **Traversals** — DFS (in/pre/post) & BFS
4. **Morris traversal** — inorder in O(1) space
5. **Deletion** — the three cases
6. **The balance problem** — why a hand-rolled BST is dangerous
7. **Self-balancing trees** — AVL rotations (and red-black, treap, B-tree)
8. **The practical answer in Python** — `bisect`, `dict`/`set`, `sortedcontainers`
9. **When to use what**
10. **Complexity cheat-sheet**


## 1. What it is

A **tree** is a hierarchy of **nodes**: one **root** at the top, each node holding children, and no cycles. The vocabulary you'll need:

- **root** — the top node; **leaf** — a node with no children.
- **height** — the longest root-to-leaf path (in edges); **depth** — distance from the root down to a node.
- **binary tree** — every node has at most two children (`left`, `right`).

The star is the **binary search tree (BST)**: a binary tree with an ordering invariant — for every node, *everything in its left subtree is smaller, everything in its right subtree is larger*. That invariant turns search into a sequence of left/right decisions: **O(log n) when the tree is balanced**... and O(n) when it isn't (section 6).

## 2. A binary search tree from scratch

`insert` and `search` both walk down from the root, branching left or right by comparing values — so each costs the tree's **height**. We write them **iteratively** (a `while` loop), both because it's faster and because a degenerate tree would otherwise blow Python's recursion limit.

In [1]:
class BSTNode:
    __slots__ = ("value", "left", "right")
    def __init__(self, value):
        self.value = value
        self.left = self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, value):                 # walk down, hang a new leaf; O(height)
        if self.root is None:
            self.root = BSTNode(value)
            return
        node = self.root
        while True:
            if value < node.value:
                if node.left is None:
                    node.left = BSTNode(value); return
                node = node.left
            elif value > node.value:
                if node.right is None:
                    node.right = BSTNode(value); return
                node = node.right
            else:
                return                       # ignore duplicates

    def __contains__(self, value):           # O(height)
        node = self.root
        while node:
            if value == node.value:
                return True
            node = node.left if value < node.value else node.right
        return False

    def height(self):                        # iterative DFS - safe on degenerate trees
        if self.root is None:
            return -1
        stack, best = [(self.root, 0)], 0
        while stack:
            node, d = stack.pop()
            best = max(best, d)
            if node.left:  stack.append((node.left,  d + 1))
            if node.right: stack.append((node.right, d + 1))
        return best

t = BST()
for v in [5, 3, 8, 1, 4, 7, 9]:
    t.insert(v)
print("7 in t :", 7 in t)
print("6 in t :", 6 in t)
print("height :", t.height(), "edges")


7 in t : True
6 in t : False
height : 2 edges


## 3. Traversals — DFS & BFS

Visiting every node comes in two families — which tie straight back to the stacks-&-queues notebook:

- **Depth-first (DFS)** — go deep before wide; naturally recursive (an implicit **stack**). Three orders, named by *when you visit a node relative to its children*:
  - **in-order** (left, node, right) → for a BST this emits values **in sorted order**.
  - **pre-order** (node, left, right) → root first; good for copying / serializing.
  - **post-order** (left, right, node) → children first; good for deleting / evaluating.
- **Breadth-first (BFS / level-order)** — visit level by level using a **queue** (`deque`).

In [2]:
from collections import deque

def in_order(node):     # left, node, right  -> sorted for a BST
    return [] if node is None else in_order(node.left) + [node.value] + in_order(node.right)

def pre_order(node):    # node, left, right
    return [] if node is None else [node.value] + pre_order(node.left) + pre_order(node.right)

def post_order(node):   # left, right, node
    return [] if node is None else post_order(node.left) + post_order(node.right) + [node.value]

def level_order(root):  # BFS with a queue
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.value)
            q.append(node.left)
            q.append(node.right)
    return out

print("in-order   :", in_order(t.root), "<- sorted")
print("pre-order  :", pre_order(t.root))
print("post-order :", post_order(t.root))
print("level-order:", level_order(t.root))


in-order   : [1, 3, 4, 5, 7, 8, 9] <- sorted
pre-order  : [5, 3, 1, 4, 8, 7, 9]
post-order : [1, 4, 3, 7, 9, 8, 5]
level-order: [5, 3, 8, 1, 4, 7, 9]


## 4. Morris traversal — inorder in O(1) space

The recursive and iterative inorder walks (section 3) both pay **O(h)** extra space — the call stack, or an explicit one — to remember the ancestors they still owe a visit. **Morris traversal** (J. H. Morris, 1979) gets the same inorder sequence in **O(1)** extra space by *borrowing* the tree's own null pointers.

The trick is **threading**: a node with a left subtree is always visited right after its **in-order predecessor** — the rightmost node of that left subtree, whose `right` pointer is null. So before descending left, we temporarily wire that predecessor's `right` to point *back* at the current node:

```
   visit B, descend left to A; A is B's predecessor, A.right is null
        B                     B   <- thread:  A.right = B
       / \          ==>      / ^
      A   C                  A_/
```

Now the walk can climb back up without a stack. On the **second** time we reach a node (arriving via a thread), we know the left subtree is finished, so we **undo the thread**, emit the node, and go right. Each edge is traversed at most twice → **O(n) time**, and the tree is left exactly as it started.

The cost: the threads briefly mutate the tree, so it isn't reentrant / thread-safe mid-walk. The payoff is real when stack space is the binding constraint (deep trees near the recursion limit — see *The balance problem* — or memory-tight environments).

In [3]:
import random

def morris_inorder(root):
    """In-order traversal in O(1) extra space by threading predecessors."""
    out, node = [], root
    while node is not None:
        if node.left is None:                     # no left subtree: visit, go right
            out.append(node.value)
            node = node.right
        else:
            pred = node.left                      # find in-order predecessor:
            while pred.right is not None and pred.right is not node:
                pred = pred.right                 #   rightmost node of the left subtree
            if pred.right is None:                # first visit: set the thread, go left
                pred.right = node
                node = node.left
            else:                                 # second visit: thread already set,
                pred.right = None                 #   undo it, visit, then go right
                out.append(node.value)
                node = node.right
    return out

# Prove it: Morris must agree with the recursive inorder over many random BSTs,
# AND must leave the tree untouched (the threads are fully undone).
rng = random.Random(0)
for _ in range(300):
    b = BST()
    for v in rng.sample(range(1000), k=rng.randint(0, 60)):
        b.insert(v)
    expected = in_order(b.root)                   # recursive, O(h) stack (section 3)
    assert morris_inorder(b.root) == expected     # O(1) space, same answer
    assert in_order(b.root) == expected           # tree fully restored afterwards
print("Morris inorder == recursive inorder over 300 random BSTs  (O(1) extra space)")

# Same threading idea yields pre-order — just visit on the FIRST touch instead.
def morris_preorder(root):
    out, node = [], root
    while node is not None:
        if node.left is None:
            out.append(node.value); node = node.right
        else:
            pred = node.left
            while pred.right is not None and pred.right is not node:
                pred = pred.right
            if pred.right is None:
                out.append(node.value)            # visit before threading -> pre-order
                pred.right = node; node = node.left
            else:
                pred.right = None; node = node.right
    return out

assert morris_preorder(t.root) == pre_order(t.root)
print("Morris preorder == recursive preorder :", morris_preorder(t.root))

Morris inorder == recursive inorder over 300 random BSTs  (O(1) extra space)
Morris preorder == recursive preorder : [5, 3, 1, 4, 8, 7, 9]


## 5. Deleting a node — the three cases

Deletion is the operation that really tests the invariant, because removing an interior node can't just leave a hole. Find the node, then handle one of three cases:

| The node has… | Do this |
|---|---|
| **no children** (a leaf) | detach it |
| **one child** | splice that child up into its place |
| **two children** | replace its value with its **in-order successor** — the smallest value in its right subtree — then delete *that* successor (which has at most one child, so it collapses to an easier case) |

The successor is the next-larger value, so swapping it in keeps everything ordered. We write it recursively, returning the new subtree root at each step so the parent re-links automatically:

In [4]:
def delete(root, value):
    """Return the (possibly new) root of the subtree with `value` removed."""
    if root is None:
        return None
    if value < root.value:
        root.left = delete(root.left, value)
    elif value > root.value:
        root.right = delete(root.right, value)
    else:                                        # found the node to remove
        if root.left is None:
            return root.right                    # 0 children, or a right child only
        if root.right is None:
            return root.left                     # a left child only
        succ = root.right                        # two children: in-order successor =
        while succ.left is not None:             #   leftmost node of the right subtree
            succ = succ.left
        root.value = succ.value                  # copy its value up...
        root.right = delete(root.right, succ.value)   # ...then delete the successor
    return root

bst = BST()
for v in [5, 3, 8, 1, 4, 7, 9]:
    bst.insert(v)
print("start                        :", in_order(bst.root))

bst.root = delete(bst.root, 4)                   # a leaf
print("after delete 4 (leaf)        :", in_order(bst.root))
bst.root = delete(bst.root, 3)                   # now has one child
print("after delete 3 (one child)   :", in_order(bst.root))
bst.root = delete(bst.root, 8)                   # two children
print("after delete 8 (two children):", in_order(bst.root))

result = in_order(bst.root)
print("still a valid BST (sorted)   :", result == sorted(result))


start                        : [1, 3, 4, 5, 7, 8, 9]
after delete 4 (leaf)        : [1, 3, 5, 7, 8, 9]
after delete 3 (one child)   : [1, 5, 7, 8, 9]
after delete 8 (two children): [1, 5, 7, 9]
still a valid BST (sorted)   : True


## 6. The balance problem

Here's the catch that makes a hand-rolled BST dangerous: its performance depends entirely on its **shape**, and the shape depends on **insertion order**. Feed it values that are already **sorted** and every new node hangs off the right of the previous one — the "tree" degenerates into a **linked list** of height n−1, and every search is O(n). Insert in random order and it stays roughly balanced at height ~log n.

In [5]:
import random, math

random.seed(0)

def height_of(values):
    b = BST()
    for v in values:
        b.insert(v)
    return b.height()

print(f"{'n':>6}{'sorted-insert height':>22}{'random height':>15}{'log2(n)':>10}")
for n in (1000, 2000, 4000):
    shuffled = list(range(n)); random.shuffle(shuffled)
    print(f"{n:>6}{height_of(range(n)):>22}{height_of(shuffled):>15}{math.log2(n):>10.1f}")


     n  sorted-insert height  random height   log2(n)
  1000                   999             20      10.0
  2000                  1999             25      11.0


  4000                  3999             27      12.0


A sorted insert gives height n−1 — a degenerate vine, O(n) per operation — while a random insert stays near log₂ n. Real systems fix this with **self-balancing** trees (AVL, red-black, B-trees) that rotate on insert to *guarantee* O(log n). **CPython ships none of them** — which is exactly why, in Python, you usually don't hand-roll a BST at all.

## 7. Self-balancing trees — AVL

Section 6 showed the danger: a sorted insert builds a degenerate vine of height n−1. A **self-balancing** BST fixes this by *rotating* after each insert so the height stays Θ(log n). The textbook example is the **AVL tree** (Adelson-Velsky & Landis, 1962): every node stores its subtree height, and the tree keeps each node's **balance factor** — `height(left) − height(right)` — within `{−1, 0, +1}`.

A **rotation** is a local, O(1) pointer rewiring that rebalances a subtree while preserving the BST ordering (the in-order sequence is unchanged):

```
      y                              x
     / \                            / \
    x   C    right-rotate(y) ->    A   y
   / \       <- left-rotate(x)        / \
  A   B                              B   C
```

After inserting, walk back up; if a node goes out of balance, one of four fixes restores it — **LL** (right-rotate), **RR** (left-rotate), **LR** (left then right), **RL** (right then left).

In [6]:
import math

class AVLNode:
    __slots__ = ("key", "left", "right", "height")
    def __init__(self, key):
        self.key = key
        self.left = self.right = None
        self.height = 1

def _h(n):  return n.height if n else 0
def _bf(n): return _h(n.left) - _h(n.right) if n else 0
def _fix_height(n): n.height = 1 + max(_h(n.left), _h(n.right))

def _rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    _fix_height(y); _fix_height(x)
    return x                                   # x is the new subtree root

def _rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    _fix_height(x); _fix_height(y)
    return y

def avl_insert(root, key):
    if root is None:
        return AVLNode(key)
    if key < root.key:
        root.left = avl_insert(root.left, key)
    elif key > root.key:
        root.right = avl_insert(root.right, key)
    else:
        return root                            # ignore duplicates
    _fix_height(root)
    bf = _bf(root)
    if bf > 1 and key < root.left.key:                       # LL
        return _rotate_right(root)
    if bf < -1 and key > root.right.key:                     # RR
        return _rotate_left(root)
    if bf > 1 and key > root.left.key:                       # LR
        root.left = _rotate_left(root.left);  return _rotate_right(root)
    if bf < -1 and key < root.right.key:                     # RL
        root.right = _rotate_right(root.right); return _rotate_left(root)
    return root

def inorder(root, out):
    if root:
        inorder(root.left, out); out.append(root.key); inorder(root.right, out)

# Feed it the exact adversarial input from section 6 — a fully sorted insert:
print(f"{'n':>6}{'AVL height':>12}{'vine height':>13}{'~1.44·log2 n':>14}")
for n in (1000, 2000, 4000):
    root = None
    for v in range(n):                         # sorted -> worst case for a naive BST
        root = avl_insert(root, v)
    out = []; inorder(root, out)
    assert out == list(range(n))               # still a valid, in-order BST
    print(f"{n:>6}{_h(root):>12}{n - 1:>13}{1.44 * math.log2(n):>14.1f}")


     n  AVL height  vine height  ~1.44·log2 n
  1000          10          999          14.4
  2000          11         1999          15.8
  4000          12         3999          17.2


**Other balancers, same idea.** Red-black trees relax the balance condition (height up to ~2·log n) so they rotate less on insert/delete — which is why `std::map`, the Linux kernel, and Java's `TreeMap` use them. A **treap** balances *probabilistically*: each node carries a random priority and stays heap-ordered on priority while BST-ordered on key, giving expected O(log n) height with far less code. **B-trees** fan out wide to stay shallow on disk (databases, filesystems). But section 6's punchline still holds: **CPython ships none of these**, so in practice you reach for `bisect` or `sortedcontainers` (next) rather than hand-rolling one.

## 8. The practical answer in Python

So what do you actually use when you need ordered, searchable data?

- **Just membership / lookup by key?** A `set` / `dict` already gives **O(1)** — only beat that when you genuinely need *order*.
- **A sorted sequence with fast search?** Keep a plain **sorted list** and use **`bisect`**: `bisect_left` / `bisect_right` binary-search in **O(log n)**, and `insort` inserts in order. The insert still shifts elements (O(n)), so `bisect` shines for **lookup-heavy, insert-light** data, range queries, and "find nearest".
- **Fast inserts *and* ordered lookups?** Reach for the third-party **`sortedcontainers`** (`SortedList` / `SortedDict` / `SortedSet`) — effectively O(log n) for both, via lists-of-lists rather than tree rotations.

In [7]:
import bisect

data = [1, 3, 4, 7, 9, 11]

i = bisect.bisect_left(data, 7)
print("index of 7       :", i, "| present:", i < len(data) and data[i] == 7)
print("where 6 would go :", bisect.bisect_left(data, 6))

bisect.insort(data, 6)                # insert keeping sorted order (O(n) shift)
print("after insort(6)  :", data)

# a range query falls out of two bisects, in O(log n + k):
lo = bisect.bisect_left(data, 4)
hi = bisect.bisect_right(data, 9)
print("values in [4, 9] :", data[lo:hi])


index of 7       : 3 | present: True
where 6 would go : 3
after insort(6)  : [1, 3, 4, 6, 7, 9, 11]
values in [4, 9] : [4, 6, 7, 9]


And `sortedcontainers` — the third-party answer when you need both fast inserts *and* sorted order — in action. A `SortedList` reorders itself on every `add` (O(log n)) and supports indexing, `bisect`, and O(1) min/max, with no balanced-tree code to maintain. (It's built from lists-of-lists, not rotations, but gives the same O(log n) profile — and it's what you'd actually reach for in Python.)

In [8]:
from sortedcontainers import SortedList, SortedDict

sl = SortedList([5, 1, 3])
sl.add(2)                              # stays sorted, O(log n) - no manual rebalancing
print("SortedList   :", list(sl), "| index of 3:", sl.bisect_left(3), "| min/max:", sl[0], sl[-1])
sl.remove(3)
print("after remove :", list(sl))

sd = SortedDict()                      # a dict that iterates in sorted key order
for k, v in [(3, "c"), (1, "a"), (2, "b")]:
    sd[k] = v
print("SortedDict   :", list(sd.items()))

SortedList   : [1, 2, 3, 5] | index of 3: 2 | min/max: 1 5
after remove : [1, 2, 5]
SortedDict   : [(1, 'a'), (2, 'b'), (3, 'c')]


## 9. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Membership / key lookup, no order | `set` / `dict` | O(1) average — unbeatable when order doesn't matter |
| Sorted data, lookup-heavy | sorted `list` + `bisect` | O(log n) search; insert O(n) but simple & stdlib |
| Sorted data, insert- *and* lookup-heavy | `sortedcontainers` (3rd-party) | ~O(log n) for both, no balancing code to maintain |
| A priority queue | `heapq` | O(log n) min; partial order only (see heaps notebook) |
| A hierarchy / parse tree / filesystem | a node class | the tree *is* the model; traverse with DFS/BFS |
| A guaranteed-balanced ordered map | `sortedcontainers` / a DB index | CPython has no built-in balanced tree |

## 10. Complexity cheat-sheet

| Operation | BST (balanced) | BST (degenerate) | sorted list + `bisect` | `dict` / `set` |
|---|:---:|:---:|:---:|:---:|
| Search | O(log n) | O(n) | O(log n) | O(1) avg |
| Insert | O(log n) | O(n) | O(n)† | O(1) avg |
| Delete | O(log n) | O(n) | O(n) | O(1) avg |
| Min / max | O(log n) | O(n) | O(1) at ends | O(n) |
| Sorted / in-order scan | O(n) | O(n) | O(n), already sorted | O(n log n), must sort |
| Range query | O(log n + k) | O(n) | O(log n + k) | O(n) |

<sub>† O(log n) to *find* the spot via `bisect`, but O(n) to shift elements over to insert it.</sub>